# Question Answering - RAG with bge-m3
This notebook implements retrieval-augmented generation for Arabic QA.

Pipeline:
1. Embed contexts with BAAI/bge-m3
2. Retrieve top-K contexts using cosine similarity on NumPy matrices
3. Construct Arabic RAG prompt
4. Generate answer using the best transformer model exported from notebook 2
5. Evaluate with BLEU and ROUGE-L

In [ ]:
import os
import re
import gc
import json
import glob
import random
import warnings
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM

from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer

warnings.filterwarnings('ignore')
print('[INFO] Imports ready for RAG notebook.')

In [ ]:
SEED = 42
TEST_SIZE = 0.2
TOP_K = 3
MAX_GEN_TOKENS = 96

DATASET_ROOT = '.'
PREPROCESSED_DIR = os.path.join(DATASET_ROOT, 'preprocessed datasets')
ORIGINAL_DATASET = os.path.join(DATASET_ROOT, 'AAFAQ_Dataset.csv')
OUTPUT_DIR = os.path.join(DATASET_ROOT, 'outputs')
TRANSFORMER_MANIFEST = os.path.join(OUTPUT_DIR, 'question_answering_transformers_best_manifest.json')
os.makedirs(OUTPUT_DIR, exist_ok=True)

BGE_MODEL_NAME = 'BAAI/bge-m3'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f'[INFO] Device: {DEVICE}')

In [ ]:
def clean_text(text: str) -> str:
    text = str(text).strip()
    return re.sub(r'\s+', ' ', text)

def build_dataset_registry() -> Dict[str, str]:
    registry = {'Original_raw': ORIGINAL_DATASET}
    if os.path.isdir(PREPROCESSED_DIR):
        for path in sorted(glob.glob(os.path.join(PREPROCESSED_DIR, '*.csv'))):
            name = os.path.splitext(os.path.basename(path))[0]
            registry[name] = path
    return registry

def load_dataset(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    required_cols = {'QuestionText', 'Answer', 'Category'}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f'Missing columns {missing} in {path}')
    df = df.dropna(subset=['QuestionText', 'Answer']).copy()
    df['QuestionText'] = df['QuestionText'].map(clean_text)
    df['Answer'] = df['Answer'].map(clean_text)
    df['Category'] = df['Category'].map(clean_text)
    df = df[(df['QuestionText'].str.len() > 0) & (df['Answer'].str.len() > 0)].reset_index(drop=True)
    return df

def compute_bleu(references: List[str], predictions: List[str]) -> float:
    smooth = SmoothingFunction().method4
    vals = []
    for r, p in zip(references, predictions):
        rt = r.split()
        pt = p.split()
        vals.append(sentence_bleu([rt], pt, smoothing_function=smooth) if pt else 0.0)
    return float(np.mean(vals)) if vals else 0.0

def compute_rouge_l(references: List[str], predictions: List[str]) -> float:
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)
    vals = [scorer.score(r, p)['rougeL'].fmeasure for r, p in zip(references, predictions)]
    return float(np.mean(vals)) if vals else 0.0

DATASET_REGISTRY = build_dataset_registry()
all_df = pd.concat([load_dataset(p).assign(_dataset=n) for n, p in DATASET_REGISTRY.items()], ignore_index=True)
train_df, test_df = train_test_split(all_df, test_size=TEST_SIZE, random_state=SEED, shuffle=True)
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)
print(f'[INFO] RAG corpus rows={len(train_df)} | eval rows={len(test_df)}')

In [ ]:
print('[INFO] Loading bge-m3 encoder...')
bge_model = SentenceTransformer(BGE_MODEL_NAME, device=str(DEVICE))

context_texts = train_df['Answer'].tolist()
context_questions = train_df['QuestionText'].tolist()
context_meta = train_df[['_dataset', 'Category']].to_dict('records')

context_vectors = bge_model.encode(context_texts, convert_to_numpy=True, show_progress_bar=True, normalize_embeddings=True)
print(f'[INFO] Context matrix shape: {context_vectors.shape}')

def retrieve_topk(question: str, k: int = TOP_K):
    q_vec = bge_model.encode([question], convert_to_numpy=True, normalize_embeddings=True)[0]
    scores = context_vectors @ q_vec
    idx = np.argsort(scores)[::-1][:k]
    rows = []
    for i in idx:
        rows.append({
            'context': context_texts[i],
            'source_question': context_questions[i],
            'score': float(scores[i]),
            'dataset': context_meta[i]['_dataset'],
            'category': context_meta[i]['Category'],
        })
    return rows

print('[INFO] Retrieval index ready.')

In [ ]:
if not os.path.exists(TRANSFORMER_MANIFEST):
    raise FileNotFoundError(f'Missing manifest from notebook 2: {TRANSFORMER_MANIFEST}')

with open(TRANSFORMER_MANIFEST, 'r', encoding='utf-8') as f:
    best_models = json.load(f)

best_models_df = pd.DataFrame(best_models)
best_models_df = best_models_df.sort_values(['TestBLEU', 'TestROUGE_L'], ascending=False).reset_index(drop=True)
best_row = best_models_df.iloc[0].to_dict()

best_model_key = best_row['Model']
best_ckpt = best_row['CheckpointPath']
hf_name = {
    'T5': 'google/mt5-small',
    'GPT': 'aubmindlab/aragpt2-base',
    'QWEN': 'Qwen/Qwen2-0.5B',
}[best_model_key]
family = 'seq2seq' if best_model_key == 'T5' else 'causal'

print(f"[INFO] Selected best model from notebook 2: {best_model_key} | ckpt={best_ckpt}")

tokenizer = AutoTokenizer.from_pretrained(hf_name)
if family == 'seq2seq':
    gen_model = AutoModelForSeq2SeqLM.from_pretrained(hf_name).to(DEVICE)
else:
    gen_model = AutoModelForCausalLM.from_pretrained(hf_name).to(DEVICE)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        gen_model.config.pad_token_id = tokenizer.pad_token_id

payload = torch.load(best_ckpt, map_location=DEVICE)
gen_model.load_state_dict(payload['model_state_dict'])
gen_model.eval()
print('[INFO] Best generator model loaded.')

In [ ]:
def build_rag_prompt(question: str, retrieved_context: str) -> str:
    return f"بناءً على السياق التالي، أجب عن السؤال.\nالسياق: {retrieved_context}\nالسؤال: {question}\nالإجابة:"

def generate_answer(prompt: str) -> str:
    enc = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512).to(DEVICE)
    with torch.no_grad():
        out = gen_model.generate(**enc, max_new_tokens=MAX_GEN_TOKENS, num_beams=4, early_stopping=True)
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    if 'الإجابة:' in text:
        text = text.split('الإجابة:')[-1].strip()
    return text

rows = []
for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc='RAG evaluation'):
    question = row['QuestionText']
    reference = row['Answer']
    retrieved = retrieve_topk(question, k=TOP_K)
    retrieved_context = ' '.join([x['context'] for x in retrieved])
    prompt = build_rag_prompt(question, retrieved_context)
    prediction = generate_answer(prompt)
    rows.append({
        'question': question,
        'reference': reference,
        'prediction': prediction,
        'retrieved_context': retrieved_context,
        'top1_score': retrieved[0]['score'] if retrieved else np.nan,
        'top1_dataset': retrieved[0]['dataset'] if retrieved else None,
        'generator_model': best_model_key,
    })

rag_df = pd.DataFrame(rows)
bleu = compute_bleu(rag_df['reference'].tolist(), rag_df['prediction'].tolist())
rouge_l = compute_rouge_l(rag_df['reference'].tolist(), rag_df['prediction'].tolist())
print(f'[RESULT] RAG BLEU={bleu:.4f} | RAG ROUGE-L={rouge_l:.4f}')

rag_summary = pd.DataFrame([{
    'Method': 'RAG_with_best_transformer',
    'GeneratorModel': best_model_key,
    'BLEU': bleu,
    'ROUGE_L': rouge_l,
    'TopK': TOP_K,
}])

baseline_summary = pd.DataFrame([{
    'Method': 'Best_plain_transformer_from_notebook2',
    'GeneratorModel': best_row['Model'],
    'BLEU': best_row['TestBLEU'],
    'ROUGE_L': best_row['TestROUGE_L'],
    'TopK': 0,
}])

comparison_df = pd.concat([baseline_summary, rag_summary], ignore_index=True)
comparison_df

In [ ]:
rag_results_path = os.path.join(OUTPUT_DIR, 'question_answering_rag_results.csv')
rag_examples_path = os.path.join(OUTPUT_DIR, 'question_answering_rag_examples.csv')
rag_compare_path = os.path.join(OUTPUT_DIR, 'question_answering_rag_comparison.csv')

rag_summary.to_csv(rag_results_path, index=False)
rag_df.to_csv(rag_examples_path, index=False)
comparison_df.to_csv(rag_compare_path, index=False)

print(f'[SAVE] {rag_results_path}')
print(f'[SAVE] {rag_examples_path}')
print(f'[SAVE] {rag_compare_path}')
display(comparison_df)

In [ ]:
seq_best_path = os.path.join(OUTPUT_DIR, 'question_answering_seq2sqe_best_models.csv')
tr_manifest_path = os.path.join(OUTPUT_DIR, 'question_answering_transformers_best_manifest.json')

global_rows = []
if os.path.exists(seq_best_path):
    seq_best_df = pd.read_csv(seq_best_path)
    for _, r in seq_best_df.iterrows():
        global_rows.append({
            'Source': 'Seq2Seq',
            'ArchitectureOrModel': r.get('Architecture', ''),
            'Dataset': r.get('Dataset', ''),
            'EmbeddingStrategy': r.get('EmbeddingStrategy', ''),
            'BLEU': r.get('TestBLEU', np.nan),
            'ROUGE_L': r.get('TestROUGE_L', np.nan),
            'CheckpointPath': r.get('CheckpointPath', ''),
        })

if os.path.exists(tr_manifest_path):
    with open(tr_manifest_path, 'r', encoding='utf-8') as f:
        tr_best = json.load(f)
    for r in tr_best:
        global_rows.append({
            'Source': 'Transformer',
            'ArchitectureOrModel': r.get('Model', ''),
            'Dataset': r.get('BestDataset', ''),
            'EmbeddingStrategy': r.get('BestEmbeddingStrategy', ''),
            'BLEU': r.get('TestBLEU', np.nan),
            'ROUGE_L': r.get('TestROUGE_L', np.nan),
            'CheckpointPath': r.get('CheckpointPath', ''),
        })

global_rows.append({
    'Source': 'RAG',
    'ArchitectureOrModel': best_model_key,
    'Dataset': 'AllCombined',
    'EmbeddingStrategy': f'bge-m3_top{TOP_K}',
    'BLEU': bleu,
    'ROUGE_L': rouge_l,
    'CheckpointPath': best_ckpt,
})

global_summary_df = pd.DataFrame(global_rows)
global_summary_path = os.path.join(OUTPUT_DIR, 'question_answering_global_summary.csv')
global_summary_df.to_csv(global_summary_path, index=False)
print(f'[SAVE] {global_summary_path}')
display(global_summary_df.head(20))